# PM-RL1 published stage results — 2026-10-02
Runs from inside a clean checkout using only standard Python and public JSON files. Checks public hashes and summary arithmetic; does not rerun models, source review, human calibration or private source reconstruction. Historical artifacts remain frozen. P3 reward qualification is pending.


In [1]:
from pathlib import Path
import hashlib, json
here = Path.cwd().resolve()
repo = next(p for p in (here, *here.parents) if (p / 'project/docs/pm_rl1_completion_20260930_v2/p3_development_results.json').is_file())
snapshot = repo / 'project/docs/reviews/20261002/pm_rl1'
def read(path):
    return json.loads(path.read_text())
manifest = read(snapshot / 'MANIFEST.json')
for row in manifest['files']:
    actual = hashlib.sha256((snapshot / row['published_path']).read_bytes()).hexdigest()
    assert actual == row['published_sha256'] == row['source_sha256']
for row in manifest['analysis_and_result_assets']:
    assert hashlib.sha256((repo / row['path']).read_bytes()).hexdigest() == row['sha256']
print(json.dumps({'copied_result_files_verified': len(manifest['files']), 'analysis_assets_verified': len(manifest['analysis_and_result_assets']), 'scope': 'Public bytes only; withheld source contents are not reverified here.'}, indent=2))


{
  "copied_result_files_verified": 26,
  "analysis_assets_verified": 16,
  "scope": "Public bytes only; withheld source contents are not reverified here."
}


In [2]:
capacity = read(snapshot / 'P1/capacity.json')
admission = read(snapshot / 'P2/admission_summary.json')
training = read(snapshot / 'P2/training_summary.json')
reload = read(snapshot / 'P2/training_reload.json')
assert sum(v['examples'] for v in admission['coverage'].values()) == admission['accepted'] == 493
assert training['frozen_base_hash_before'] == training['frozen_base_hash_after']
assert reload['passed'] and reload['absolute_nll_delta'] == 0
print(json.dumps({'P1_prefixes': capacity['prefixes'], 'P1_reachable_plans': capacity['total_reachable'], 'P2_weak_targets': admission['accepted'], 'P2_optimizer_steps': training['steps'], 'baseline_dev_nll': training['baseline_dev_nll'], 'selected_dev_nll': reload['dev']['nll'], 'limitations': ['Four accepted targets affected by duplicate-key admission defect; all 493 remain the actual training set.', 'NLL and coverage do not establish support-quality gains.', 'Only two of six cross-load repeated replies matched exactly; root cause unresolved.']}, indent=2))


{
  "P1_prefixes": 114,
  "P1_reachable_plans": 6899,
  "P2_weak_targets": 493,
  "P2_optimizer_steps": 78,
  "baseline_dev_nll": 1.786513342472241,
  "selected_dev_nll": 1.1173423809777827,
  "limitations": [
    "Four accepted targets affected by duplicate-key admission defect; all 493 remain the actual training set.",
    "NLL and coverage do not establish support-quality gains.",
    "Only two of six cross-load repeated replies matched exactly; root cause unresolved."
  ]
}


In [3]:
p3 = read(repo / 'project/docs/pm_rl1_completion_20260930_v2/p3_development_results.json')
dev = p3['development']
assert dev['physical_J_calls'] == dev['first_attempts'] + dev['semantic_recheck_path_probes'] == 33
assert not dev['reward_qualified'] and p3['human_instrument']['labels_received'] == 0
assert dev['sealed_validation_calls'] == dev['independent_repeat_draws_run'] == 0
throughput = p3['throughput']
assert abs(throughput['engine_wall_seconds'] / dev['physical_J_calls'] - throughput['warm_batched_engine_seconds_per_finished_job']) < 1e-9
print(json.dumps({'P3_development_J': dev['physical_J_calls'], 'API_usd': dev['API_usd'], 'engine_minutes': throughput['engine_wall_seconds'] / 60, 'output_tokens_median_including_thinking': throughput['output_tokens_including_thinking']['median'], 'human_pairs_prepared': p3['human_instrument']['active_pairs'], 'human_labels_received': p3['human_instrument']['labels_received'], 'reward_qualified': dev['reward_qualified'], 'forecast_scope': 'J engine only, assuming measured four-way throughput transfers; not a complete 96-hour schedule.', 'forecasts': throughput['forecasts']}, indent=2))


{
  "P3_development_J": 33,
  "API_usd": 0,
  "engine_minutes": 29.17641474508176,
  "output_tokens_median_including_thinking": 5994,
  "human_pairs_prepared": 40,
  "human_labels_received": 0,
  "reward_qualified": false,
  "forecast_scope": "J engine only, assuming measured four-way throughput transfers; not a complete 96-hour schedule.",
  "forecasts": [
    {
      "episodes_per_seed": 128,
      "remaining_quality_slots_envelope": 3258,
      "remaining_semantic_and_technical_extra_J_envelope": 248,
      "warm_batched_J_hours_if_sample_rate_transfers": 51.66288388699831,
      "individual_request_seconds_linear_hours": 188.1231842549849,
      "optional_RQX_J_hours_in_batched_envelope": 4.2438421447391645
    },
    {
      "episodes_per_seed": 384,
      "remaining_quality_slots_envelope": 4794,
      "remaining_semantic_and_technical_extra_J_envelope": 248,
      "warm_batched_J_hours_if_sample_rate_transfers": 74.29670865894052,
      "individual_request_seconds_linear_hours":